In [75]:
import duckdb
import polars as pl
from pathlib import Path

## Last.fm data

In [76]:
data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)
print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df.head()

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
"""Yung Lean""","""""","""Frost God""","""""","""Hennessy & Sailor Moon""","""""",1739471209,2025-02-13 18:26:49
"""Cynthoni""","""ef8dae4f-8b71-454f-ad40-9f5370…","""Sewerslvt Presents: Cynthoni, …","""2e902803-9553-4c37-92f3-e0ed26…","""Lychee Ice""","""ea05e66c-6664-445e-b7f2-bc46ad…",1724575008,2024-08-25 08:36:48
"""Dikke Baap""","""""","""IK TRIP 'M (Dimitri K Remix)""","""""","""IK TRIP 'M - Dimitri K Remix""","""""",1746971994,2025-05-11 13:59:54
"""Pikekyss""","""96180fc2-a8b6-466c-86a7-b621ff…","""Herregud""","""""","""Herregud""","""""",1686306507,2023-06-09 10:28:27
"""TØFL""","""21a3df18-4cb5-478f-8955-fdcfb4…","""Alle Andre Får Lov""","""""","""Alle andre får lov""","""af5abc9a-1a51-445b-838b-c6cbe3…",1646590577,2022-03-06 18:16:17


In [77]:
df.filter(pl.col("artist_name").is_null())

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
null,"""""",null,"""""",null,"""""",1740484061,2025-02-25 11:47:41
null,"""""",null,"""""",null,"""""",1740507046,2025-02-25 18:10:46
null,"""""",null,"""""",null,"""""",1740159957,2025-02-21 17:45:57
null,"""""",null,"""""",null,"""""",1740590547,2025-02-26 17:22:27
null,"""""",null,"""""",null,"""""",1740558905,2025-02-26 08:35:05
…,…,…,…,…,…,…,…
null,"""""",null,"""""",null,"""""",1740580554,2025-02-26 14:35:54
null,"""""",null,"""""",null,"""""",1740519868,2025-02-25 21:44:28
null,"""""",null,"""""",null,"""""",1740506479,2025-02-25 18:01:19


In [78]:
earliest_timestamp = df.select(pl.col("date_played_unix").min()).item()
earliest_timestamp

1621491935

## Spotify data

In [79]:
spotify_dir = Path("../data/spotify")
spotify_db = duckdb.read_json(spotify_dir/"extended streaming history/*.json")
spotify_df = spotify_db.pl()
spotify_df.head()

ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,audiobook_title,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
datetime[μs],str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool,i64,bool
2017-12-17 10:31:16,"""Partner spotify web_player""",216391,"""ZZ""","""84.209.176.247""","""Like A G6""","""Far East Movement""","""Free Wired""","""spotify:track:5AyL2kgLtTWEu3qO…",null,null,null,null,null,null,null,"""clickrow""","""unknown""",false,false,false,null,false
2017-12-17 10:34:47,"""Partner spotify web_player""",210942,"""ZZ""","""84.209.176.247""","""Rocketeer""","""Far East Movement""","""Free Wired""","""spotify:track:3ZuiIFm9nRnkRz8A…",null,null,null,null,null,null,null,"""unknown""","""unknown""",false,false,false,null,false
2017-12-17 10:38:45,"""Partner spotify web_player""",236946,"""ZZ""","""84.209.176.247""","""Live My Life""","""Far East Movement""","""Dirty Bass""","""spotify:track:6dpKQiQzZE2r9rZV…",null,null,null,null,null,null,null,"""unknown""","""unknown""",false,false,false,null,false
2017-12-17 10:41:56,"""Partner spotify web_player""",189949,"""ZZ""","""84.209.176.247""","""Freal Luv (feat. Chanyeol & Ti…","""Far East Movement""","""Freal Luv (feat. Chanyeol & Ti…","""spotify:track:00rbUJGbcjw3FW5c…",null,null,null,null,null,null,null,"""unknown""","""unknown""",false,false,false,null,false
2017-12-17 10:43:01,"""Partner spotify web_player""",63316,"""ZZ""","""84.209.176.247""","""Turn Up The Love""","""Far East Movement""","""Dirty Bass""","""spotify:track:4HQ9gXAtONKs8NCM…",null,null,null,null,null,null,null,"""unknown""","""trackdone""",false,false,false,null,false


In [80]:
columns_of_interest = [
    "ts",
    "ms_played",
    "master_metadata_track_name",
    "master_metadata_album_artist_name",
    "master_metadata_album_album_name",
    "offline_timestamp",
    "spotify_track_uri"
]

new_col_names = {
    'ts': "track_played_utc",
    'master_metadata_track_name': "track_name",
    'master_metadata_album_artist_name': "artist_name",
    'master_metadata_album_album_name': "album_name",
    'offline_timestamp': "date_played_unix",
}

spotify_df = (
    spotify_df
    .select(columns_of_interest)
    .rename(new_col_names)
)
spotify_df

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
datetime[μs],i64,str,str,str,i64,str
2017-12-17 10:31:16,216391,"""Like A G6""","""Far East Movement""","""Free Wired""",null,"""spotify:track:5AyL2kgLtTWEu3qO…"
2017-12-17 10:34:47,210942,"""Rocketeer""","""Far East Movement""","""Free Wired""",null,"""spotify:track:3ZuiIFm9nRnkRz8A…"
2017-12-17 10:38:45,236946,"""Live My Life""","""Far East Movement""","""Dirty Bass""",null,"""spotify:track:6dpKQiQzZE2r9rZV…"
2017-12-17 10:41:56,189949,"""Freal Luv (feat. Chanyeol & Ti…","""Far East Movement""","""Freal Luv (feat. Chanyeol & Ti…",null,"""spotify:track:00rbUJGbcjw3FW5c…"
2017-12-17 10:43:01,63316,"""Turn Up The Love""","""Far East Movement""","""Dirty Bass""",null,"""spotify:track:4HQ9gXAtONKs8NCM…"
…,…,…,…,…,…,…
2025-09-29 07:56:14,671,"""Love Takes Miles""","""Cameron Winter""","""Love Takes Miles""",null,"""spotify:track:48cPTTDcZz6PRE1k…"
2025-09-29 15:05:25,20810,"""take me by the hand (feat. Bla…","""Oklou""","""take me by the hand (feat. Bla…",1759158304,"""spotify:track:0nuH0isoVlLDIRk3…"
2025-10-01 06:43:34,5200,"""The Coward Forgets His Nightma…","""Chanel Beads""","""The Coward Forgets His Nightma…",1759301009,"""spotify:track:3LItdxCkntoyFyoh…"


In [81]:
# From the resulting dataframe I see that there may be null-values. Lets investigate further.
spotify_df.null_count()

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
u32,u32,u32,u32,u32,u32,u32
0,0,4967,4967,4967,220564,4967


Consistent missing values across 4 out of 7 which indicate some systemic pattern.

`date_played_unix` missing timestamps is not a problem as we can use 
`track_played_utc` to get the UNIX-timestamps.

In [82]:
spotify_df.filter(pl.col("track_name").is_null())

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
datetime[μs],i64,str,str,str,i64,str
2018-03-29 19:01:25,708162,null,null,null,null,null
2018-05-31 11:19:20,1384536,null,null,null,null,null
2018-08-17 16:46:03,2340000,null,null,null,null,null
2019-10-09 15:28:12,1314017,null,null,null,null,null
2019-10-09 17:45:42,4200,null,null,null,null,null
…,…,…,…,…,…,…
2025-06-10 12:44:29,0,null,null,null,1749559469,null
2025-06-10 13:35:43,2090,null,null,null,1749562540,null
2025-06-10 13:35:50,60,null,null,null,1749562550,null


In [83]:
spotify_df = spotify_df.filter(pl.col("track_name").is_not_null())
spotify_df

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
datetime[μs],i64,str,str,str,i64,str
2017-12-17 10:31:16,216391,"""Like A G6""","""Far East Movement""","""Free Wired""",null,"""spotify:track:5AyL2kgLtTWEu3qO…"
2017-12-17 10:34:47,210942,"""Rocketeer""","""Far East Movement""","""Free Wired""",null,"""spotify:track:3ZuiIFm9nRnkRz8A…"
2017-12-17 10:38:45,236946,"""Live My Life""","""Far East Movement""","""Dirty Bass""",null,"""spotify:track:6dpKQiQzZE2r9rZV…"
2017-12-17 10:41:56,189949,"""Freal Luv (feat. Chanyeol & Ti…","""Far East Movement""","""Freal Luv (feat. Chanyeol & Ti…",null,"""spotify:track:00rbUJGbcjw3FW5c…"
2017-12-17 10:43:01,63316,"""Turn Up The Love""","""Far East Movement""","""Dirty Bass""",null,"""spotify:track:4HQ9gXAtONKs8NCM…"
…,…,…,…,…,…,…
2025-09-29 07:56:14,671,"""Love Takes Miles""","""Cameron Winter""","""Love Takes Miles""",null,"""spotify:track:48cPTTDcZz6PRE1k…"
2025-09-29 15:05:25,20810,"""take me by the hand (feat. Bla…","""Oklou""","""take me by the hand (feat. Bla…",1759158304,"""spotify:track:0nuH0isoVlLDIRk3…"
2025-10-01 06:43:34,5200,"""The Coward Forgets His Nightma…","""Chanel Beads""","""The Coward Forgets His Nightma…",1759301009,"""spotify:track:3LItdxCkntoyFyoh…"


In [84]:
# Filling in missing timestamps using 'date_played_unix'
spotify_df = ( 
    spotify_df
    .with_columns(
        date_played_unix = pl.col("date_played_unix").fill_null(
            pl.col("track_played_utc").dt.strftime("%s").cast(int)
        )
    )
)
spotify_df

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
datetime[μs],i64,str,str,str,i64,str
2017-12-17 10:31:16,216391,"""Like A G6""","""Far East Movement""","""Free Wired""",1513506676,"""spotify:track:5AyL2kgLtTWEu3qO…"
2017-12-17 10:34:47,210942,"""Rocketeer""","""Far East Movement""","""Free Wired""",1513506887,"""spotify:track:3ZuiIFm9nRnkRz8A…"
2017-12-17 10:38:45,236946,"""Live My Life""","""Far East Movement""","""Dirty Bass""",1513507125,"""spotify:track:6dpKQiQzZE2r9rZV…"
2017-12-17 10:41:56,189949,"""Freal Luv (feat. Chanyeol & Ti…","""Far East Movement""","""Freal Luv (feat. Chanyeol & Ti…",1513507316,"""spotify:track:00rbUJGbcjw3FW5c…"
2017-12-17 10:43:01,63316,"""Turn Up The Love""","""Far East Movement""","""Dirty Bass""",1513507381,"""spotify:track:4HQ9gXAtONKs8NCM…"
…,…,…,…,…,…,…
2025-09-29 07:56:14,671,"""Love Takes Miles""","""Cameron Winter""","""Love Takes Miles""",1759132574,"""spotify:track:48cPTTDcZz6PRE1k…"
2025-09-29 15:05:25,20810,"""take me by the hand (feat. Bla…","""Oklou""","""take me by the hand (feat. Bla…",1759158304,"""spotify:track:0nuH0isoVlLDIRk3…"
2025-10-01 06:43:34,5200,"""The Coward Forgets His Nightma…","""Chanel Beads""","""The Coward Forgets His Nightma…",1759301009,"""spotify:track:3LItdxCkntoyFyoh…"


In [85]:
# Filter the dataset to the time range of interest.
# Then apply a filter to keep "valid scrobbles", as the exported dataset
# tracks any song played regardless of duration played.
spotify_df = ( 
    spotify_df
    .filter(pl.col("date_played_unix") < earliest_timestamp) 
    .filter(pl.col("ms_played") >= 30000)
    .drop(pl.col("ms_played"))
)
spotify_df

track_played_utc,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
datetime[μs],str,str,str,i64,str
2017-12-17 10:31:16,"""Like A G6""","""Far East Movement""","""Free Wired""",1513506676,"""spotify:track:5AyL2kgLtTWEu3qO…"
2017-12-17 10:34:47,"""Rocketeer""","""Far East Movement""","""Free Wired""",1513506887,"""spotify:track:3ZuiIFm9nRnkRz8A…"
2017-12-17 10:38:45,"""Live My Life""","""Far East Movement""","""Dirty Bass""",1513507125,"""spotify:track:6dpKQiQzZE2r9rZV…"
2017-12-17 10:41:56,"""Freal Luv (feat. Chanyeol & Ti…","""Far East Movement""","""Freal Luv (feat. Chanyeol & Ti…",1513507316,"""spotify:track:00rbUJGbcjw3FW5c…"
2017-12-17 10:43:01,"""Turn Up The Love""","""Far East Movement""","""Dirty Bass""",1513507381,"""spotify:track:4HQ9gXAtONKs8NCM…"
…,…,…,…,…,…
2021-05-20 06:11:42,"""Xcxoplex (with Charli XCX)""","""A. G. Cook""","""Xcxoplex (with Charli XCX)""",1621491102,"""spotify:track:3Qtc8RGzuzbDGBpZ…"
2021-05-20 06:15:39,"""Miss The Rage (feat. Playboi C…","""Trippie Redd""","""Miss The Rage (feat. Playboi C…",1621491339,"""spotify:track:2BITQ360Knh6qNAO…"
2021-05-20 06:18:03,"""The Kiss Of Venus (Dominic Fik…","""Paul McCartney""","""McCartney III Imagined""",1621491483,"""spotify:track:28kOGtTZzbfQ8fMm…"


In [86]:
# Time to export the data 
from pathlib import Path

dir_path = Path("../data/spotify")
spotify_df.write_parquet(dir_path/"2017-2021 spotify data.parquet")

## Vstack attempt

In [69]:
spotify_df.head()

track_played_utc,track_name,artist_name,album_name,date_played_unix,spotify_track_uri
datetime[μs],str,str,str,i64,str
2017-12-17 10:31:16,"""Like A G6""","""Far East Movement""","""Free Wired""",1513506676,"""spotify:track:5AyL2kgLtTWEu3qO…"
2017-12-17 10:34:47,"""Rocketeer""","""Far East Movement""","""Free Wired""",1513506887,"""spotify:track:3ZuiIFm9nRnkRz8A…"
2017-12-17 10:38:45,"""Live My Life""","""Far East Movement""","""Dirty Bass""",1513507125,"""spotify:track:6dpKQiQzZE2r9rZV…"
2017-12-17 10:41:56,"""Freal Luv (feat. Chanyeol & Ti…","""Far East Movement""","""Freal Luv (feat. Chanyeol & Ti…",1513507316,"""spotify:track:00rbUJGbcjw3FW5c…"
2017-12-17 10:43:01,"""Turn Up The Love""","""Far East Movement""","""Dirty Bass""",1513507381,"""spotify:track:4HQ9gXAtONKs8NCM…"


In [93]:
set(spotify_df.columns + df.columns)

{'album_mbid',
 'album_name',
 'artist_mbid',
 'artist_name',
 'date_played_unix',
 'spotify_track_uri',
 'track_mbid',
 'track_name',
 'track_played_utc'}

In [92]:
sorted(spotify_df.columns)

['album_name',
 'artist_name',
 'date_played_unix',
 'spotify_track_uri',
 'track_name',
 'track_played_utc']

In [70]:
spotify_df = spotify_df.with_columns(
    artist_mbid = pl.lit(""),
    album_mbid = pl.lit(""),
    track_mbid = pl.lit("")
)

In [71]:
df.head()

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc
str,str,str,str,str,str,i64,datetime[μs]
"""Target Demographic""","""""","""Dreaming About You""","""""","""Dreaming About You""","""""",1737548117,2025-01-22 12:15:17
"""Tyler, The Creator""","""""","""IGOR""","""""","""I THINK""","""""",1738923412,2025-02-07 10:16:52
"""Frank Ocean""","""e520459c-dff4-491d-a6e4-c97be3…","""Blonde""","""08f54f68-7c89-4e22-8a0f-ac2b06…","""Solo (Reprise)""","""8682b979-399c-4b50-a220-926042…",1669365610,2022-11-25 08:40:10
"""Teen Suicide""","""a63a5c27-aae4-40b3-86c7-527496…","""i will be my own hell because …","""2559fbc2-0b36-48aa-a2fd-1b36eb…","""the same things happening to m…","""9e193cae-5f5a-42af-adfd-959024…",1655985482,2022-06-23 11:58:02
"""Mitski""","""fa58cf24-0e44-421d-8519-8bf461…","""The Land Is Inhospitable and S…","""65c45a70-b218-46d0-8dfc-4983b2…","""My Love Mine All Mine""","""b293285f-5600-4380-ae13-792406…",1694941251,2023-09-17 09:00:51


In [72]:
df = df.with_columns(
    spotify_track_uri = pl.lit("")
)
cols = df.columns

In [73]:
df = df.select(cols)
spotify_df = spotify_df.select(cols)
df_full = spotify_df.vstack(df).sort(pl.col("date_played_unix"), descending=False)
df_full

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,spotify_track_uri
str,str,str,str,str,str,i64,datetime[μs],str
"""Far East Movement""","""""","""Free Wired""","""""","""Like A G6""","""""",1513506676,2017-12-17 10:31:16,"""spotify:track:5AyL2kgLtTWEu3qO…"
"""Far East Movement""","""""","""Free Wired""","""""","""Rocketeer""","""""",1513506887,2017-12-17 10:34:47,"""spotify:track:3ZuiIFm9nRnkRz8A…"
"""Far East Movement""","""""","""Dirty Bass""","""""","""Live My Life""","""""",1513507125,2017-12-17 10:38:45,"""spotify:track:6dpKQiQzZE2r9rZV…"
"""Far East Movement""","""""","""Freal Luv (feat. Chanyeol & Ti…","""""","""Freal Luv (feat. Chanyeol & Ti…","""""",1513507316,2017-12-17 10:41:56,"""spotify:track:00rbUJGbcjw3FW5c…"
"""Far East Movement""","""""","""Dirty Bass""","""""","""Turn Up The Love""","""""",1513507381,2017-12-17 10:43:01,"""spotify:track:4HQ9gXAtONKs8NCM…"
…,…,…,…,…,…,…,…,…
"""brakence""","""""","""punk2""","""bad4bc48-356b-45fe-b8ad-d2b897…","""ginger tea""","""c0645055-8b90-4e90-8222-f09b80…",1774462214,2026-03-25 18:10:14,""""""
"""Maya Ricci""","""""","""Syko""","""""","""Syko""","""""",1774462362,2026-03-25 18:12:42,""""""
"""Yeat""","""9e6a6a2f-f696-4cc0-87f1-e4f712…","""Alivë""","""""","""Diëd B4""","""""",1774462486,2026-03-25 18:14:46,""""""


In [94]:
df_full.null_count()

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,spotify_track_uri
u32,u32,u32,u32,u32,u32,u32,u32,u32
17,0,17,0,17,0,0,0,0


In [95]:
df_full.filter(pl.col("artist_name").is_not_null())

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,spotify_track_uri
str,str,str,str,str,str,i64,datetime[μs],str
"""Far East Movement""","""""","""Free Wired""","""""","""Like A G6""","""""",1513506676,2017-12-17 10:31:16,"""spotify:track:5AyL2kgLtTWEu3qO…"
"""Far East Movement""","""""","""Free Wired""","""""","""Rocketeer""","""""",1513506887,2017-12-17 10:34:47,"""spotify:track:3ZuiIFm9nRnkRz8A…"
"""Far East Movement""","""""","""Dirty Bass""","""""","""Live My Life""","""""",1513507125,2017-12-17 10:38:45,"""spotify:track:6dpKQiQzZE2r9rZV…"
"""Far East Movement""","""""","""Freal Luv (feat. Chanyeol & Ti…","""""","""Freal Luv (feat. Chanyeol & Ti…","""""",1513507316,2017-12-17 10:41:56,"""spotify:track:00rbUJGbcjw3FW5c…"
"""Far East Movement""","""""","""Dirty Bass""","""""","""Turn Up The Love""","""""",1513507381,2017-12-17 10:43:01,"""spotify:track:4HQ9gXAtONKs8NCM…"
…,…,…,…,…,…,…,…,…
"""brakence""","""""","""punk2""","""bad4bc48-356b-45fe-b8ad-d2b897…","""ginger tea""","""c0645055-8b90-4e90-8222-f09b80…",1774462214,2026-03-25 18:10:14,""""""
"""Maya Ricci""","""""","""Syko""","""""","""Syko""","""""",1774462362,2026-03-25 18:12:42,""""""
"""Yeat""","""9e6a6a2f-f696-4cc0-87f1-e4f712…","""Alivë""","""""","""Diëd B4""","""""",1774462486,2026-03-25 18:14:46,""""""
